# Importing system files

### Make sure you are using the same python version as the env

In [41]:
import sys, platform
print(sys.executable)
print(platform.python_version())

/opt/miniconda3/envs/geospatial_env/bin/python
3.14.3


## Importing necessary packages

In [42]:
# Libraries
import pandas as pd
import numpy as np
import geopandas as gpd
from pathlib import Path
import os

# Statsmodels
import statsmodels.api as sm

# Metrics
from sklearn.metrics import mean_squared_error, mean_absolute_error


# Checking the src directory

In [43]:
CURRENT_DIRECTORY =  os.getcwd()
SRC_DIRECTORY = Path(CURRENT_DIRECTORY).parents[1]
print(SRC_DIRECTORY)

/Users/axlee/Desktop/Singhealth/AED-OHCA/src


# Importing datasets

In [44]:
BASE_DATASET_PATH = Path(SRC_DIRECTORY).parents[0]
BASE_DATASET_PATH = BASE_DATASET_PATH / "datasets"
print(BASE_DATASET_PATH)

/Users/axlee/Desktop/Singhealth/AED-OHCA/datasets


In [45]:
read_from_filepath = Path(BASE_DATASET_PATH / f"singapore_data/cleaned_data/ohca_data/ohca_binary_complete.csv")
final_model_df = pd.read_csv(read_from_filepath)
final_model_df = final_model_df.drop(columns=['Unnamed: 0'])

# Checking columns of the dataset

In [46]:
final_model_df.columns.tolist()

['pln_area_n',
 'subzone_n',
 'year',
 'month',
 'incident_count',
 'ohca_binary',
 'total',
 'above_60_proportion',
 'male_chinese_proportion',
 'female_chinese_proportion',
 'male_malays_proportion',
 'female_malays_proportion',
 'male_indians_proportion',
 'female_indians_proportion',
 'male_others_proportion',
 'female_others_proportion',
 'business_1_encoding',
 'business_2_encoding',
 'business_park_encoding',
 'school_encoding',
 'airport',
 'is_residential']

## Creating predictor columns


In [47]:
predictor_cols = [
       'year', 'month',
       'above_60_proportion', 'male_chinese_proportion',
       'female_chinese_proportion', 'male_malays_proportion',
       'female_malays_proportion', 'male_indians_proportion',
       'female_indians_proportion', 'male_others_proportion',
       'female_others_proportion', 'business_1_encoding',
       'business_2_encoding', 'business_park_encoding', 'school_encoding',
       'airport', 'is_residential'
]

# Keep a reference dataframe for testing/comparison
comparison_df = final_model_df[['pln_area_n', 'subzone_n', 'year', 'month', 'ohca_binary']].copy()
comparison_df

,pln_area_n,subzone_n,year,month,ohca_binary
0,ang mo kio,ang mo kio town centre,2010,4,0
1,ang mo kio,ang mo kio town centre,2010,5,1
2,ang mo kio,ang mo kio town centre,2010,6,0
3,ang mo kio,ang mo kio town centre,2010,7,1
4,ang mo kio,ang mo kio town centre,2010,8,0
...,...,...,...,...,...
43564,western water catchment,murai,2021,8,0
43565,western water catchment,murai,2021,9,1
43566,western water catchment,murai,2021,10,1
43567,western water catchment,murai,2021,11,1


# Implementing Poisson_Distribution_Model

In [48]:
# Chronological Split
# Training: 2010 - 2019
train_df = final_model_df[final_model_df['year'] <= 2019]

# Offset added to account for different "exposure" levels.
train_df['log_total'] = np.log(train_df['total'] + 1)
X_train = train_df[predictor_cols].fillna(0)
y_train = train_df['incident_count']

# Validation: 2020
val_df = final_model_df[final_model_df['year'] == 2020]

# Offset added to account for different "exposure" levels.
val_df['log_total'] = np.log(val_df['total'] + 1)
X_val = val_df[predictor_cols].fillna(0)
y_val = val_df['incident_count']

# Testing: 2021
test_df = final_model_df[final_model_df['year'] == 2021]

# Offset added to account for different "exposure" levels.
test_df['log_total'] = np.log(test_df['total'] + 1)
X_test = test_df[predictor_cols].fillna(0)
y_test = test_df['incident_count']

Model Rationale: Population Offset


Standard Poisson regression predicts the absolute number of events. However, subzones with larger populations naturally have a higher "exposure" to OHCA risk.

By including log(total_population) as an offset, the model is adjusted to predict the incidence rate rather than just the raw count, allowing for fair comparison between high-density and low-density areas.

In [49]:
# Add a constant (intercept) to the predictors
X_train_sm = sm.add_constant(X_train)

# Fit the basic Poisson model with the population offset
poisson_model = sm.GLM(y_train, X_train_sm, 
                       offset=train_df['log_total'], 
                       family=sm.families.Poisson()).fit()
print(poisson_model.summary())

                 Generalized Linear Model Regression Results                  
Dep. Variable:         incident_count   No. Observations:                36153
Model:                            GLM   Df Residuals:                    36135
Model Family:                 Poisson   Df Model:                           17
Link Function:                    Log   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -32755.
Date:                Wed, 04 Mar 2026   Deviance:                       37282.
Time:                        10:34:52   Pearson chi2:                 4.67e+07
No. Iterations:                     6   Pseudo R-squ. (CS):             0.3570
Covariance Type:            nonrobust                                         
                                coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------------------
const                 

# Zero-inflated Poisson

In [52]:
# Fit the ZIP model using the log_total offset for the count component
inflation_cols = ['is_residential', 'airport']

X_train_zip = sm.add_constant(X_train)
X_infl_zip = sm.add_constant(train_df[inflation_cols])

zip_model = sm.ZeroInflatedPoisson(endog=y_train, 
                                   exog=X_train_zip, 
                                   exog_infl=X_infl_zip, 
                                   offset=train_df['log_total'], 
                                   inflation='logit').fit(maxiter=100)

print(zip_model.summary())

/opt/miniconda3/envs/geospatial_env/lib/python3.14/site-packages/statsmodels/discrete/discrete_model.py:1331: RuntimeWarning: overflow encountered in exp
  return -np.exp(XB) +  endog*XB - gammaln(endog+1)
/opt/miniconda3/envs/geospatial_env/lib/python3.14/site-packages/statsmodels/discrete/discrete_model.py:1508: RuntimeWarning: overflow encountered in exp
  L = np.exp(np.dot(X,params) + offset + exposure)
/opt/miniconda3/envs/geospatial_env/lib/python3.14/site-packages/statsmodels/discrete/discrete_model.py:1509: RuntimeWarning: invalid value encountered in multiply
  return (self.endog - L)[:,None] * X
/opt/miniconda3/envs/geospatial_env/lib/python3.14/site-packages/statsmodels/discrete/discrete_model.py:1074: RuntimeWarning: overflow encountered in exp
  return np.exp(linpred)
/opt/miniconda3/envs/geospatial_env/lib/python3.14/site-packages/statsmodels/discrete/count_model.py:269: RuntimeWarning: invalid value encountered in multiply
  dldp[zero_idx,:] = (score_main[zero_idx].T *
/

         Current function value: 366.783382
         Iterations: 2
         Function evaluations: 32
         Gradient evaluations: 14


/opt/miniconda3/envs/geospatial_env/lib/python3.14/site-packages/statsmodels/base/model.py:595: HessianInversionWarning: Inverting hessian failed, no bse or cov_params available
  warnings.warn('Inverting hessian failed, no bse or cov_params '
/opt/miniconda3/envs/geospatial_env/lib/python3.14/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


                     ZeroInflatedPoisson Regression Results                    
Dep. Variable:          incident_count   No. Observations:                36153
Model:             ZeroInflatedPoisson   Df Residuals:                    36135
Method:                            MLE   Df Model:                           17
Date:                 Thu, 05 Mar 2026   Pseudo R-squ.:                  -329.5
Time:                         10:16:31   Log-Likelihood:            -1.3260e+07
converged:                       False   LL-Null:                       -40119.
Covariance Type:             nonrobust   LLR p-value:                     1.000
                                coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------------------
inflate_const                -1.4560        nan        nan        nan         nan         nan
inflate_is_residential       -1.0983        nan        nan        nan         

# Metrics to check

In [ ]:
# Example for test set (Standard Poisson)
test_preds = poisson_model.predict(sm.add_constant(X_test,has_constant='add'))
print(f"Poisson RMSE: {np.sqrt(mean_squared_error(y_test, test_preds))}")
print(f"Poisson MAE: {mean_absolute_error(y_test, test_preds)}")

# Example for test set (ZIP)
test_preds_zip = zip_model.predict(sm.add_constant(X_test, has_constant='add'),exog_infl=sm.add_constant(test_df[inflation_cols], has_constant='add'))
print(f"ZIP RMSE: {np.sqrt(mean_squared_error(y_test, test_preds_zip))}")
print(f"ZIP MAE: {mean_absolute_error(y_test, test_preds_zip)}")

Poisson RMSE: 1.8078316161861518
Poisson MAE: 0.9728386060509489
ZIP RMSE: 1.8085546856072077
ZIP MAE: 0.95410095806431


# Comparison between Standard Poisson and Zero Inflated Poisson

**Standard Poisson Model**
- RMSE: ~1.8078
- MAE: ~0.9728

**Zero-Inflated Poisson (ZIP) Model**
- RMSE: ~1.8085
- MAE: ~0.9541

While the Zero-Inflated Poisson model yielded a slightly better (lower) Mean Absolute Error (MAE), indicating it handles median errors a bit better by explicitly modeling "certain zero" subzones (like airports or non-residential areas), it performed marginally worse on the Root Mean Squared Error (RMSE). 

Because RMSE heavily penalizes larger errors and predicting high-incident counts accurately is critical for anticipating out-of-hospital cardiac arrest (OHCA) hotspots, the added complexity of the zero-inflated mechanism did not yield drastically better predictive performance across the board for this specific dataset. A standard Poisson model (or potentially a Negative Binomial model to handle overdispersion) may be more robust and simpler to implement.

# Export final predictions into clean CSV

#

In [54]:
# 1. Prepare the full dataset (all years) just like the training data
X_all = final_model_df[predictor_cols].fillna(0)
X_all_sm = sm.add_constant(X_all, has_constant='add')

# Ensure the offset exists for the full dataframe
final_model_df['log_total'] = np.log(final_model_df['total'] + 1)

# 2. Generate predictions using your best model (e.g., poisson_model)
# Note: If zip_model performed better, change this to zip_model.predict(X_all, exog_infl=final_model_df[inflation_cols])
final_model_df['predicted_incident_count'] = poisson_model.predict(X_all_sm, offset=final_model_df['log_total'])

# 3. Save as a clean CSV
export_csv_path = Path(BASE_DATASET_PATH / "singapore_data/cleaned_data/ohca_data/final_poisson_predictions.csv")
final_model_df.to_csv(export_csv_path, index=False)

print(f"Predictions successfully exported to: {export_csv_path}")

Predictions successfully exported to: /Users/axlee/Desktop/Singhealth/AED-OHCA/datasets/singapore_data/cleaned_data/ohca_data/final_poisson_predictions.csv


# Export as GeoPackage (.gpkg) for QGIS visualization

In [ ]:
# 1. Load the subzone geometry
subzones_2019_filepath = Path(BASE_DATASET_PATH / "singapore_data/data_gov/masterplan_2019/subzone_boundary_2019.gpkg")
subzone_2019_gdf = gpd.read_file(subzones_2019_filepath)

# Standardize names for merging
subzone_2019_gdf.columns = subzone_2019_gdf.columns.str.lower()
subzone_2019_gdf['subzone_n'] = subzone_2019_gdf['subzone_n'].str.strip().str.lower()
subzone_2019_gdf['pln_area_n'] = subzone_2019_gdf['pln_area_n'].str.strip().str.lower()

# 2. Filter predictions to just the most recent year (2021) and calculate the total predicted per subzone
preds_2021 = final_model_df[final_model_df['year'] == 2021]
yearly_preds_2021 = preds_2021.groupby(['pln_area_n', 'subzone_n'])['predicted_incident_count'].sum().reset_index()

# 3. Merge the predictions onto the map geometry
gdf_predictions = subzone_2019_gdf.merge(yearly_preds_2021, on=['subzone_n', 'pln_area_n'], how='left')

# 4. Export to GeoPackage
export_gpkg_path = Path(BASE_DATASET_PATH / "singapore_data/cleaned_data/ohca_data/poisson_predictions_2021_map.gpkg")
gdf_predictions.to_file(export_gpkg_path, driver="GPKG")

print(f"GeoPackage successfully exported to: {export_gpkg_path}")

GeoPackage successfully exported to: /Users/axlee/Desktop/Singhealth/AED-OHCA/datasets/singapore_data/cleaned_data/ohca_data/poisson_predictions_2021_map.gpkg
